## Configuration

In [ ]:
from pathlib import Path
import tomllib
from utils.utils import *
import json

PATH_EEG = Path("../EEG/EEG/results/")
FILENAME = 'Case_05_ExG.csv'
MARKER_FILE = 'Case_05_Marker.csv'
CONFIG_PATH = "../EEG/EEG/config/config.toml"

with open(CONFIG_PATH, "rb") as f:
    config = tomllib.load(f)

BANDS = config['BANDS']
SEGMENT_DEF = config['SEGMENT_DEF']
RANDOM_SEED = config['RANDOM_SEED']
DURATION = config['DURATION']

SFREQ = config['SFREQ']
L_FREQ = config['L_FREQ']
H_FREQ = config['H_FREQ']

## Raw Data

In [ ]:
raw_full, df_full, df_markers_full = load_and_preprocess_eeg(PATH_EEG, FILENAME, MARKER_FILE, SFREQ, L_FREQ, H_FREQ)

raws, labels = extract_segments(raw_full, df_full, df_markers_full, SEGMENT_DEF)

In [ ]:
plot_psd_comparison(raws, labels)

In [ ]:
plot_topomap_comparison(raws, labels, band_name='Alpha')
plot_topomap_comparison(raws, labels, band_name='Beta')

In [ ]:
plot_topomap_comparison(raws, labels, band_name='Delta')
plot_topomap_comparison(raws, labels, band_name='Theta')

## Preprocessing

In [ ]:
import mne
from mne.preprocessing import ICA
raw, df, df_markers = load_and_preprocess_eeg(PATH_EEG, FILENAME, MARKER_FILE, SFREQ, L_FREQ, H_FREQ)

# ICA
# n_components=15 (16 Channels, ICA finding max n_channels - 1 or same number)
ica = ICA(n_components=15, max_iter='auto', random_state=RANDOM_SEED)

print(">>> ICA (Might take a while)...")
ica.fit(raw)

print(">>> Showing Components...")
ica.plot_components()
plt.show()

ica.plot_sources(raw, show_scrollbars=True)
plt.show()

In [ ]:
exclude_indices = [0, 1, 2, 3, 4, 5]

print(f">>> Removing ingredients: {exclude_indices}")
ica.exclude = exclude_indices

raw_clean = raw.copy()
ica.apply(raw_clean)

print(">>> Cleaned data.")

target_chan = "F3" if "F3" in raw.ch_names else raw.ch_names[0]

print(f"Comparison on the channel {target_chan}...")
fig, ax = plt.subplots(figsize=(15, 6))
ax.plot(raw.get_data(picks=target_chan)[0, 1000:2500], color='red', alpha=0.5, label='Before cleaning (Original)')
ax.plot(raw_clean.get_data(picks=target_chan)[0, 1000:2500], color='black', label='After cleaning (Clean)')
ax.set_title(f"Artifact Removal Effect (Eyes and Muscles) - Channel {target_chan}")
ax.legend()
plt.show()

In [ ]:
raws, labels = extract_segments(raw_clean, df_full, df_markers_full, SEGMENT_DEF)

plot_psd_comparison(raws, labels)

In [ ]:
plot_topomap_comparison(raws, labels, band_name='Alpha')
plot_topomap_comparison(raws, labels, band_name='Beta')

In [ ]:
plot_topomap_comparison(raws, labels, band_name='Delta')
plot_topomap_comparison(raws, labels, band_name='Gamma')

#### Analiza Pasma Alpha (8–12 Hz):
W trybie pasywnym (Brainrot Static) obszar P5 (lewa kora skroniowo-ciemieniowa) płonie na ciemnoczerwono. Podobnie dzieje się w trybie Smart Static. Czerwień w Alfie to stan silnej synchronizacji (uśpienia/wyłączenia danego obszaru). Oznacza to, że gdy ten pacjent tylko "patrzy" (nie dotyka ekranu), jego lewy ośrodek odpowiedzialny m.in. za integrację sensoryczno-językową całkowicie się "odcina". Reszta mózgu (P3, C3) jest wybudzona (niebieska).

Efekt Scrolla: W trybie Brainrot Scroll ta czerwona plama w P5 nagle znika (robi się biało-niebieska).
Fizyczny ruch przewijania szybkiego wideo (Brainrot Scroll) jest jedynym bodźcem, który potrafi "wybudzić" tę uśpioną część lewej półkuli.

#### Analiza Pasma Beta (13–30 Hz):
We wszystkich trybach utrzymuje się silna aktywność (czerwień) w lewym tylnym rejonie (P5). Lewa strona mocno pracuje nad analizą detali.

W trybie Smart Static prawa kora ruchowa/czuciowa (C4) jest biała (neutralna). Natomiast gdy pacjent wchodzi w Brainrot Scroll, nagle na C4 pojawia się wyraźna lekko czerwona plama.

Przewijanie "głupot" (Brainrot) silnie aktywuje jego prawą półkulę czuciowo-ruchową (C4). Prawa półkula odpowiada za przetwarzanie holistyczne i orientację przestrzenną. Oznacza to, że przewijanie to dla niego intensywne doświadczenie przestrzenno-motoryczne.

#### Analiza Pasma Delta (1–4 Hz):
Obserwacja: Ciemnoczerwona, bardzo gęsta plama na lewym biegunie (P5) jest niezwykle silna w trybie Brainrot Static, a w trybach Smart (Static i Scroll) przesuwa się lekko w dół ku potylicy (O1).

Wysoka Delta w stanie czuwania to często sygnał znużenia poznawczego lub awaryjnego odcięcia. Lewy analizator wzrokowo-językowy jest permanentnie przemęczony lub celowo tłumiony przez układ nerwowy podczas konsumpcji tych treści.

#### Analiza Pasma Gamma (>30 Hz):
W trybie Brainrot Scroll widać wyrzut Gammy. Jest tam nie tylko ciemnoczerwona plama z lewej strony (P5), ale też czerwony pożar z prawej strony (wokół C4).

Brainrot Scroll to dla mózgu obciążenie obliczeniowe. Musi integrować (Gamma) lewopółkulowe bodźce (dźwięk/tekst z wideo) z prawopółkulową nawigacją przestrzenną (C4). To kosztuje sporo energii.

## Spektogram

In [ ]:
# Alpha (fatigue/relax)
plot_band_power_over_time(raws, labels, band=(8, 12), band_name='Alpha')
# Beta (Focus)
plot_band_power_over_time(raws, labels, band=(13, 30), band_name='Beta')

In [ ]:
# Emotion
calculate_asymmetry(raws, labels, ch_left='F3', ch_right='F4')

# According to Heller's model, parietal asymmetry is responsible for physiological arousal
calculate_asymmetry(raws, labels, ch_left='P3', ch_right='P4')

# Image processing method.
calculate_asymmetry(raws, labels, ch_left='O1', ch_right='O2')

# Motor
calculate_asymmetry(raws, labels, ch_left='C3', ch_right='C4')

### Podsumowanie

#### Kora Czołowa (F4 vs F3)
**Brainrot (Pasywny i Scroll)**: Słupki są na plusie. To oznacza lewopółkulową motywację dążeniową (zaangażowanie, pozytywny afekt). Ten wykres dowodzi, że badanemu to ogromne przebodźcowanie sprawia przyjemność.

**Smart (Pasywny i Scroll)**: Słupki drastycznie spadają poniżej zera. Kiedy tylko treść jest bardziej smart, mózg odczuwa znużenie, niechęć i chęć wycofania się (withdrawal). To fizjologiczny dowód na odrzucenie treści edukacyjnych.

#### Kora Ciemieniowa (P4 vs P3)
Minusy we wszystkich trybach.

Dominacja prawej kory ciemieniowej (P4) oznacza zaangażowanie w orientację przestrzenną, często połączone z niepokojem lub czujnością. Aplikacja utrzymuje badanego w ciągłym, bardzo silnym napięciu w tle.

#### Kora Potyliczna (O2 vs O1)
Wszystkie słupki są dodatnie, co w przypadku asymetrii oznacza dominację lewej kory wzrokowej (O1).

Pacjent nie patrzy na obrazki jako na całość (holistycznie). On cały czas "skanuje" detale. Co ciekawe, asymetria ta jest największa w trybach Smart. Próbuje tam czytać tekst (lewa półkula). W trybie pasywnego Brainrotu wskaźnik spada blisko zera.

#### Kora Ruchowa (C4 vs C3)
Wszystkie słupki są głęboko na minusie. Ujemny wynik oznacza silną dominację prawej kory ruchowej (C4).

Ten pacjent przewijał ekran lewą ręką (lub był tak mocno zorientowany na prawopółkulowe przetwarzanie przestrzenne interfejsu, że C4 nieustannie pracowało). W trybie Brainrot Scroll słupek jest mniejszy niż w reszcie trybów.